In [1]:
# ...existing code...
"""
Refactored test runner for extract_cophieu68.
Functions:
 - init_env(): chuẩn bị sys.path
 - init_config(): khởi tạo ETLPipelineConfig
 - build_crawler(): tạo instance crawler an toàn
 - get_test_symbols(): lấy danh sách symbols (config -> market list -> fallback)
 - safe_serialize(): serialize kết quả cho JSON
 - run_symbol_tests(): chạy các test per-symbol
 - run_non_symbol_tests(): chạy các test không cần symbol
 - save_summary(): lưu file JSON tóm tắt
 - main(): chạy toàn bộ
Paste whole cell vào notebook và chạy.
"""


# PROJECT_ROOT = "/mnt/c/Users/Admin/Downloads/Project/Github/ETL_Project"

# def init_env():
#     if PROJECT_ROOT not in sys.path:
#         sys.path.insert(0, PROJECT_ROOT)

# # imports that depend on project path
# init_env()
import time
from internal.dags.cophieu68_dag.extract.extract_cophieu68 import ExtractCophieu68
from internal.dags.ETL_Orchestra.main_orchestra_etl import ETLPipelineConfig
from internal.dags.cophieu68_dag.load.base_loading import *
from internal.models.cophieu68_model.extract_models import *
config_path = "/mnt/c/Users/Admin/Downloads/Project/Github/ETL_Project/internal/config/web_craw_config/cophieu68_config.yaml"
config = ETLPipelineConfig(config_path=config_path)
pipeline_config = config.config
extract_pipeline_logger = config.etl_extract_logger
crawler = ExtractCophieu68(config.config, config.etl_extract_logger)
# set attributes mà extract_cophieu68.__init__ thường tạo
try:
    crawler.endpoint = crawler.crawler_cfg.get("endpoints", {})
except Exception:
    crawler.endpoint = {}



In [2]:
from internal.dags.cophieu68_dag.load.load_datalake_cophieu68 import *
mongo_config =config.config.get("storage", {}).get("mongodb", {})
loading_datalake = MongoLoader(
    username = mongo_config.get("username", ""),
    password = mongo_config.get("password", ""),
    host = mongo_config.get("host", "localhost"),
    authSource = mongo_config.get("authSource", "admin"),
    port = mongo_config.get("port", 27017),
    database = mongo_config.get("database", "ETL_Project"),
)
backend_mongo =MongoStorageBackend(loading_datalake)

In [3]:
list_stock_collection = mongo_config.get("collections", {}).get("list_stock", "list_stock")
trading_data_collection = mongo_config.get("collections", {}).get("trading_data", "trading_data")
symbol_data = backend_mongo.find_table(name = list_stock_collection)
symbol_info = {}
for item in symbol_data["data"]:
    if item["market_type"] == "VNINDEX":
        symbol_list = item["symbol_list"]
symbol_list = [item["symbol_list"] for item in symbol_data["data"] if item["market_type"] == "VNINDEX"]
for item in symbol_list[0][:1]:
    crawl_trading_data = crawler.crawl_trading_data(symbol = item)
    loading_datalake.load_trading_data(collection = trading_data_collection, trading_data = crawl_trading_data)
    time.sleep(1)  

{"time":"2025-11-12T17:05:39", "level":"INFO", "message":"Hết dữ liệu ở trang 42", "caller":"/mnt/c/Users/Admin/Downloads/Project/Github/ETL_Project/internal/dags/cophieu68_dag/extract/extract_cophieu68.py:420"}


Exception: Error loading trading data into MongoDB: Modifiers operate on fields but we found type string instead. For example: {$mod: {<field>: ...}} not {$set: "VCB"}, full error: {'index': 0, 'code': 9, 'errmsg': 'Modifiers operate on fields but we found type string instead. For example: {$mod: {<field>: ...}} not {$set: "VCB"}'}

In [ ]:
# print(crawl_trading_data)

In [ ]:
# import time
# list_stock_collection = mongo_config.get("collections", {}).get("list_stock", "list_stock")
# company_profile_collection = mongo_config.get("collections", {}).get("stock_info", "company_profile")
# symbol_data = backend_mongo.find_table(name = list_stock_collection)
# symbol_info = {}
# for item in symbol_data["data"]:
#     if item["market_type"] == "VNINDEX":
#         symbol_list = item["symbol_list"]
# symbol_list = [item["symbol_list"] for item in symbol_data["data"] if item["market_type"] == "VNINDEX"]
# for symbol in symbol_list[0]:
#     company_profile = crawler.crawl_company_profile(symbol)
#     symbol_info[symbol] = company_profile.__dict__
#     time.sleep(0.25)

# loading_datalake.load_company_info(
#     collection_name=company_profile_collection,
#     company_profiles=symbol_info
# )

In [ ]:
# industry_list_collection = mongo_config.get("collections", {}).get("industry_list", "industry_list")
# industry_info = {}
# for key in INDUSTRIAL_INFO_TYPE:
#     industry_list = crawler.crawl_industry_info(type_info = key)
#     industry_info[key] = industry_list
# print(industry_info)
# loading_datalake.load_crawl_industry_info(
#     collection_name=industry_list_collection,
#     industry_data=industry_info
# )

In [ ]:
# stock_lists = []
# for key in CRAWL_MARKET_LIST_CONFIG:
#     stock_list_market_type = crawler.crawl_market_list(key)
#     stock_lists.append(stock_list_market_type)
# list_stock_collection = mongo_config.get("collections", {}).get("list_stock", "list_stock")
# print(stock_lists)
# loading_datalake.load_market_list(
#     collection_name=list_stock_collection,
#     stock_info=stock_lists
# )